# 11.1 - Agent Fundamentals: The Loop That Acts

**Module 11 - AI Agents** | Netsetos GenAI Engineering

This notebook builds an LLM agent from first principles: the think -> act -> observe loop, in about forty lines of plain Python, plus the real Anthropic and Google SDK versions it maps onto. You will classify chatbot vs workflow vs agent, watch the loop select tools and reason (ReAct) before acting, and see why reliability compounds (`p^N`) so you cap iterations and reach for an agent only when the task truly needs one.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

Install the SDKs this lesson touches. Most cells run as pure-Python simulations with no key needed; only the two real-API cells (Anthropic and Google) require a provider key. Uncomment the pip line on Colab or a fresh environment.

In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install "anthropic>=0.40.0" google-genai python-dotenv -q  # noqa


**Explanation**

Environment prep, not agent code. It installs the Anthropic SDK, the new `google-genai` SDK, and `python-dotenv` so the later real-API cells have their dependencies.

**How the code works - step by step**
- **`anthropic>=0.40.0`** - the Anthropic Messages API client used by the real tool-use loop cell.
- **`google-genai`** - the current Google SDK (replaces the deprecated `google.generativeai`) used by the one-flag agent cell.
- **`python-dotenv`** - lets you load API keys from a `.env` file instead of hardcoding them.

**In one line:** install the two provider SDKs plus dotenv; the simulation cells need none of them.

**What you'll see:** (no output - environment prep)

## API keys

Before any real model call, keys must be in the environment. The simulation cells (1-2, 5-8) run without a key; only the Anthropic and Google cells need one. Any single provider key is enough to start.

> **Needs a provider key for the real-API cells later** (set `ANTHROPIC_API_KEY` and/or `GOOGLE_API_KEY`).

In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("ANTHROPIC_API_KEY", "")    # console.anthropic.com
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


**Explanation**

A key-loading cell, not a model call. It seeds the environment variables the SDKs read, defaulting them to empty strings so nothing crashes on import if a key is absent.

**How the code works - step by step**
- **`os.environ.setdefault("ANTHROPIC_API_KEY", "")`** - registers the Anthropic key slot without overwriting one you already exported.
- **`os.environ.setdefault("GOOGLE_API_KEY", "")`** - same for the Google key.
- Keys come from `console.anthropic.com` and `aistudio.google.com`; they are read from the environment, never hardcoded.

**In one line:** register empty key slots so the SDKs can pick up real keys from the environment or a `.env` file.

**What you'll see:** (no output - environment prep)

## 1 - Classify: chatbot, workflow, or agent

The word 'agent' is overused, so pin it down with the one line that divides the three: **who drives the control flow?** A chatbot has no loop, a workflow runs a path you fixed, and an agent lets the LLM pick the next action and loop on the result. This function encodes exactly that test.

In [ ]:
# WHO DRIVES THE CONTROL FLOW? - the one line between chatbot, workflow, and agent.
def classify(has_tools, llm_drives_control_flow, loops_on_results):
    if not has_tools and not loops_on_results:
        return "chatbot",  "reactive Q&A; no tools, no loop"
    if has_tools and not llm_drives_control_flow:
        return "workflow", "tools on a path YOU fixed; the LLM fills steps, you drive"
    if has_tools and llm_drives_control_flow and loops_on_results:
        return "agent",    "the LLM picks the next action and loops on the result"
    return "copilot", "suggests; the human decides and acts"

systems = [
    ("a plain support bot",       False, False, False),
    ("a fixed RAG pipeline",      True,  False, False),
    ("a tool-using research bot", True,  True,  True),
]
for name, tools, drives, loops in systems:
    kind, why = classify(tools, drives, loops)
    print(f"  {name:26s} -> {kind:8s} ({why})")

# Output:
#   a plain support bot        -> chatbot  (reactive Q&A; no tools, no loop)
#   a fixed RAG pipeline       -> workflow (tools on a path YOU fixed; the LLM fills steps, you drive)
#   a tool-using research bot  -> agent    (the LLM picks the next action and loops on the result)

**Explanation**

A tiny decision function that turns three yes/no facts about a system into its label. The whole point is that a single input - whether the LLM drives the control flow - is what flips 'workflow' into 'agent'; everything else is supporting detail.

**How the code works - step by step**
- **`classify(has_tools, llm_drives_control_flow, loops_on_results)`** - branches on the three flags in priority order.
- **no tools and no loop** -> `chatbot` (reactive Q&A).
- **tools but the LLM does not drive** -> `workflow` (you fixed the path, the model fills steps).
- **tools + LLM drives + loops on results** -> `agent` (the model picks the next action).
- Anything else falls through to `copilot` (suggests; a human acts).
- The `systems` list runs three examples through it and prints each label with its reason.

**In one line:** the flag `llm_drives_control_flow` is the hinge between a workflow and an agent.

**What you'll see:** Three lines mapping each example to its kind: the support bot -> chatbot, the fixed RAG pipeline -> workflow, and the tool-using research bot -> agent, each with a one-clause explanation.

## 2 - The agent loop from scratch

Here is the whole idea in about twenty lines: **THINK** (decide the next step from task + history), **ACT** (call the chosen tool), **OBSERVE** (feed the result back), repeat until a final answer. A scripted stand-in plays the model's THINK step so the loop runs with no API key - but the shape is exactly what a real agent does.

In [ ]:
# THE AGENT LOOP FROM SCRATCH - observe -> think -> act, in ~20 lines (fake-LLM sim).
KB = {"genai": "GenAI course: 14999 INR, 146 hrs",
      "refund": "100% within 7 days, 50% within 30",
      "schedule": "Live class 7 PM IST daily"}
def search(q): return KB.get(q.lower().split()[0], "not found")
def calculate(expr): return round(eval(expr, {"__builtins__": {}}), 2)
TOOLS = {"search": search, "calculate": calculate}

def fake_llm(task, history):
    "A SCRIPTED stand-in for the LLM's THINK step (a real model decides this)."
    t = task.lower(); used = [h["tool"] + ":" + h["arg"] for h in history]
    if "gst" in t or "18%" in t:
        if "calculate:14999*1.18" not in used: return {"act": ("calculate", "14999*1.18")}
        return {"answer": f"14999 with 18% GST = {history[-1]['result']} INR"}
    if "refund" in t and ("class" in t or "live" in t):
        if "search:refund" not in used:   return {"act": ("search", "refund")}
        if "search:schedule" not in used: return {"act": ("search", "schedule")}
        return {"answer": f"Refund: {history[0]['result']}. Live class: {history[1]['result']}"}
    return {"answer": "I do not know."}

def agent(task, max_steps=5):
    history = []
    for step in range(max_steps):
        decision = fake_llm(task, history)                       # THINK
        if "answer" in decision:
            print(f"    step {step+1}: ANSWER -> {decision['answer']}"); return
        tool, arg = decision["act"]                              # ACT
        result = TOOLS[tool](arg)
        history.append({"tool": tool, "arg": arg, "result": result})   # OBSERVE
        print(f"    step {step+1}: ACT {tool}({arg}) -> OBSERVE {result}")

for task in ["What is 14999 with 18% GST?", "Refund policy and when is the live class?"]:
    print(f"  Task: {task}"); agent(task)

# Output:
#   Task: What is 14999 with 18% GST?
#     step 1: ACT calculate(14999*1.18) -> OBSERVE 17698.82
#     step 2: ANSWER -> 14999 with 18% GST = 17698.82 INR
#   Task: Refund policy and when is the live class?
#     step 1: ACT search(refund) -> OBSERVE 100% within 7 days, 50% within 30
#     step 2: ACT search(schedule) -> OBSERVE Live class 7 PM IST daily
#     step 3: ANSWER -> Refund: 100% within 7 days, 50% within 30. Live class: Live class 7 PM IST daily

**Explanation**

The core lesson cell: a runnable agent loop where `fake_llm` is a hand-scripted stand-in for the model's decision, so you can watch the control flow without a key. It handles a one-tool task (arithmetic) and a two-tool task (two lookups then combine), proving the same loop covers both.

**How the code works - step by step**
- **`KB` + `search` / `calculate`** - a three-entry knowledge base and two real tools; `TOOLS` maps names to functions.
- **`fake_llm(task, history)`** - the scripted THINK step: it inspects the task keywords and the tools already used, then returns either an `act` (tool + arg) or a final `answer`.
- **`agent(task, max_steps=5)`** - the loop itself: call `fake_llm` (THINK), if it returned an answer stop, else run the tool (ACT), append `{tool, arg, result}` to `history` (OBSERVE), and repeat.
- **`history`** - the list threaded through every turn is the agent's working memory (revisited in cell 7).
- **`max_steps`** - caps the loop so a stuck agent cannot spin forever (the reliability rule from cell 6).

**In one line:** think -> act -> observe on a growing history until the model answers.

**What you'll see:** For the GST task: one ACT `calculate(14999*1.18)` -> OBSERVE `17698.82`, then the answer. For the two-part task: ACT `search(refund)`, ACT `search(schedule)`, then a combined answer - showing single- and multi-step loops in the same code.

## 3 - Tool selection: the model reads the descriptions

How does the agent choose *which* tool? It reads each tool's **description** (name + what it does + typed args) and matches it to the task, then fills the arguments. A vague description is how a model picks the wrong tool - the description is all it has to go on.

In [ ]:
# TOOL SELECTION - the model picks WHICH tool + args by reading the tool descriptions.
TOOL_DESCS = {"search": "look up course info, refunds, schedules",
              "calculate": "arithmetic like 14999*1.18",
              "send_email": "send an email to a user"}
def select_tool(task):
    t = task.lower()
    if any(w in t for w in ["*", "+", "gst", "%", "total"]): return ("calculate", "14999*1.18")
    if any(w in t for w in ["price", "refund", "schedule", "how much"]): return ("search", "genai")
    if "email" in t or "receipt" in t: return ("send_email", "asha@example.com")
    return (None, None)

for task in ["What is 14999 with 18% GST?", "How much is the GenAI course?", "Email the receipt to Asha"]:
    tool, arg = select_tool(task)
    print(f"  {task:38s} -> {tool}({arg!r})")

# Output:
#   What is 14999 with 18% GST?            -> calculate('14999*1.18')
#   How much is the GenAI course?          -> search('genai')
#   Email the receipt to Asha              -> send_email('asha@example.com')

**Explanation**

A stand-in for the model's tool-picking step: a keyword matcher that mimics reading tool descriptions and returns the chosen tool plus an argument. It exists to make the selection step concrete and to show that a wrong pick simply wastes a turn.

**How the code works - step by step**
- **`TOOL_DESCS`** - the descriptions the model would read: search for lookups, calculate for arithmetic, send_email to message a user.
- **`select_tool(task)`** - matches task keywords to a tool: math symbols/`gst`/`%` -> `calculate`; price/refund/schedule words -> `search`; email/receipt -> `send_email`.
- Returns `(None, None)` when nothing matches, standing in for 'no suitable tool.'
- The loop prints the chosen tool and argument for three sample tasks.

**In one line:** match the task to the tool its description advertises, then fill the argument.

**What you'll see:** Three lines: the GST task -> `calculate('14999*1.18')`, the price question -> `search('genai')`, and the receipt task -> `send_email('asha@example.com')`.

## 4 - The real loop: Anthropic tool use driven by stop_reason

In production the loop is structured, not text-parsing. The response comes back with **`stop_reason == "tool_use"`** and a `tool_use` block; your code runs the tool, appends a **`tool_result`**, and loops - ending when `stop_reason` becomes **`end_turn`**. This is the same think/act/observe loop as the simulation, now model-driven.

> **Needs an Anthropic API key** (set `ANTHROPIC_API_KEY`).

In [ ]:
# THE REAL LOOP - Anthropic Messages API tool use; stop_reason drives the while.
import anthropic
client = anthropic.Anthropic()                       # reads ANTHROPIC_API_KEY

TOOLS = [{"name": "calculate", "description": "Do arithmetic, e.g. 14999*1.18.",
          "input_schema": {"type": "object", "properties": {"expr": {"type": "string"}},
                           "required": ["expr"]}}]
def calculate(expr): return round(eval(expr, {"__builtins__": {}}), 2)

messages = [{"role": "user", "content": "What is 14999 with 18% GST?"}]
max_steps = 5                                        # cap iterations - the reliability rule from Step 5
for step in range(max_steps):
    resp = client.messages.create(model="claude-sonnet-4-6", max_tokens=1024,
                                  tools=TOOLS, messages=messages)
    if resp.stop_reason != "tool_use":               # not tool_use: stop. end_turn is clean; max_tokens/refusal land here too
        print(resp.content[0].text); break
    messages.append({"role": "assistant", "content": resp.content})
    results = []
    for block in resp.content:
        if block.type == "tool_use":                 # ACT: the model asked for a tool
            out = calculate(**block.input)
            results.append({"type": "tool_result", "tool_use_id": block.id, "content": str(out)})
    messages.append({"role": "user", "content": results})   # OBSERVE: feed results back

# Output: (illustrative minimal example - needs `pip install anthropic` + a key; in production add
#   a request timeout= and wrap the call in try/except) the SAME think->act->observe loop as the sim
#   above, but the model decides and stop_reason (tool_use vs end_turn) drives it.

**Explanation**

The genuine article: the Anthropic Messages API tool-use loop where `stop_reason` drives the `while`. It mirrors the from-scratch simulation but the model - not a scripted stub - decides each step, and message-history bookkeeping (assistant `tool_use` block, then user `tool_result`) is what keeps the conversation coherent.

**How the code works - step by step**
- **`client = anthropic.Anthropic()`** - reads `ANTHROPIC_API_KEY` from the environment.
- **`TOOLS`** - one tool, `calculate`, declared with a JSON `input_schema` (this is the 10.2-style schema the model reads).
- **loop with `max_steps=5`** - the same iteration cap as the sim.
- **`resp.stop_reason != "tool_use"`** - if the model is done (`end_turn`), print the text and break; `max_tokens` and `refusal` also land here and mean stop-and-report, not another tool call.
- **append assistant `content`** - store the model's `tool_use` request.
- **run each `tool_use` block** - ACT: call `calculate(**block.input)`, collect a `tool_result` keyed by `block.id`.
- **append results as a user message** - OBSERVE: feed the tool outputs back, then loop.

**In one line:** `stop_reason` (tool_use vs end_turn) is the real agent loop's condition; tool use is two round trips - the model asks, you execute, the model answers.

**What you'll see:** Illustrative - with a valid key it prints the model's final answer (14999 with 18% GST is 17698.82). The `# Output:` comment notes this is the same think->act->observe loop as the sim, now driven by the model and `stop_reason`, and that production code should add a request timeout and try/except.

## 5 - ReAct: reason, then act

Left to call tools blindly, a model grabs a familiar-looking one and often picks wrong. **ReAct** interleaves reasoning and action: write a **Thought** ('this is arithmetic, so calculate') before each **Action**, then record the **Observation**. That cheap reasoning step catches the wrong choice before the agent acts on it.

In [ ]:
# ReAct - write a Thought BEFORE the Action; reasoning catches the wrong tool.
def blind(task):                       # no reasoning: grab a familiar tool
    return "search"
def react(task):                       # reason first, then pick
    thought = "The task is arithmetic, so I should CALCULATE, not search."
    return thought, "calculate"

task = "What is 14999 with 18% GST?"
print(f"  Task: {task}")
print(f"  BLIND -> Action: {blind(task)}  (wrong: search cannot do math -> 'not found')")
th, act = react(task)
print(f"  ReAct -> Thought: {th}")
print(f"        -> Action:  {act}  (right)")

# Output:
#   Task: What is 14999 with 18% GST?
#   BLIND -> Action: search  (wrong: search cannot do math -> 'not found')
#   ReAct -> Thought: The task is arithmetic, so I should CALCULATE, not search.
#         -> Action:  calculate  (right)

**Explanation**

A side-by-side contrast, not a real model call: a `blind` agent that grabs a tool with no reasoning versus a `react` agent that writes a Thought first. It isolates the single behavior - reasoning before acting - that makes tool use reliable.

**How the code works - step by step**
- **`blind(task)`** - returns `search` unconditionally, standing in for grabbing a familiar tool with no thought.
- **`react(task)`** - first produces a `thought` naming the task as arithmetic, then returns the correct tool `calculate`.
- The print block runs both on the GST task and labels the blind pick wrong (search cannot do math) and the ReAct pick right.

**In one line:** a Thought before the Action is the shown work that catches the wrong tool before it runs.

**What you'll see:** The task, then two traces: BLIND -> Action `search` (wrong, cannot do math), and ReAct -> Thought (names it arithmetic) -> Action `calculate` (right).

## 6 - Reliability compounds (p^N)

An agent stops naturally on a final answer, but you also need a hard `max_steps` cap because of the single most important fact in agent engineering: **reliability compounds**. If each step succeeds with probability `p`, an `N`-step run succeeds with roughly `p^N` - errors multiply, they do not average out.

In [ ]:
# RELIABILITY COMPOUNDS - each step is ~95%, but the run multiplies (p^N).
def reliability(p, n): return round(p ** n, 3)
print("  per-step success p = 0.95:")
for n in [1, 3, 5, 10, 15]:
    bar = "#" * int(reliability(0.95, n) * 20)
    print(f"    {n:>2} steps -> {reliability(0.95, n):.2f}  {bar}")

# Output:
#   per-step success p = 0.95:
#      1 steps -> 0.95  ###################
#      3 steps -> 0.86  #################
#      5 steps -> 0.77  ###############
#     10 steps -> 0.60  ###########
#     15 steps -> 0.46  #########

**Explanation**

A measurement cell, not a model call: it computes `p^N` for a fixed per-step reliability and draws a bar chart in text. The point is visceral - watch the success rate collapse as steps pile up, which is the whole argument for capping iterations and checkpointing costly steps.

**How the code works - step by step**
- **`reliability(p, n)`** - returns `p ** n`, the end-to-end success of an n-step run.
- The loop evaluates it at `p = 0.95` for `n` in 1, 3, 5, 10, 15.
- **`bar = "#" * int(...* 20)`** - scales each result to a 20-char bar so the decay is visible.

**In one line:** multiply per-step reliability across N steps and the run's success rate falls off a cliff.

**What you'll see:** A small chart at p=0.95: 1 step -> 0.95, 3 -> 0.86, 5 -> 0.77, 10 -> 0.60, 15 -> 0.46, with shrinking `#` bars making the compounding drop obvious.

## 7 - Memory: the working set

Where does an agent remember things mid-task? In the **history** the loop accumulates - the running list of actions and observations you feed back each turn. That history *is* short-term memory, it lives in the context window, and it grows every turn, so context and cost grow with it.

In [ ]:
# MEMORY - the in-loop history IS the short-term memory; it grows every turn.
def context_tokens(system, task, turns, per_turn=40):
    return system + task + per_turn * turns
sys_t, task_t = 200, 30
print("  history (and context, and cost) grows each turn:")
for turn in range(1, 6):
    print(f"    after turn {turn}: history={turn} entries, ~{context_tokens(sys_t, task_t, turn)} tokens")

# Output:
#   history (and context, and cost) grows each turn:
#     after turn 1: history=1 entries, ~270 tokens
#     after turn 2: history=2 entries, ~310 tokens
#     after turn 3: history=3 entries, ~350 tokens
#     after turn 4: history=4 entries, ~390 tokens
#     after turn 5: history=5 entries, ~430 tokens

**Explanation**

A cost/growth model, not a model call: a simple token estimator that shows the running history expanding linearly turn by turn. It makes concrete why memory is engineered rather than assumed - nothing persists automatically, and the window is finite.

**How the code works - step by step**
- **`context_tokens(system, task, turns, per_turn=40)`** - estimates total context as system + task + a flat cost per accumulated turn.
- Fixed `sys_t=200`, `task_t=30`; the loop runs turns 1 through 5.
- Each iteration prints the history size and the estimated token count, which climbs by `per_turn` every turn.

**In one line:** the in-loop history is the short-term memory, and it (plus the bill) grows linearly every turn.

**What you'll see:** Five lines showing history growing from 1 to 5 entries and context rising from ~270 to ~430 tokens - the linear growth that eventually forces you to summarize or externalize to long-term memory (Lesson 11.4).

## 8 - Agent, workflow, or just an API call?

The most valuable agent skill is knowing when NOT to build one. Reach for an agent only when the path is open-ended and the model must decide it at runtime; a single deterministic operation is one API call, and a fixed pipeline is a workflow. This function encodes that decision.

In [ ]:
# AGENT, WORKFLOW, OR JUST AN API CALL? - a decision function.
def decide(single_shot, deterministic, needs_runtime_flexibility):
    if single_shot: return "just call the API once (no loop needed)"
    if deterministic and not needs_runtime_flexibility: return "use a WORKFLOW (you fix the steps)"
    if needs_runtime_flexibility: return "use an AGENT (the LLM decides the path)"
    return "start with a workflow; add agency only if it is needed"

cases = [
    ("translate one sentence",              True,  True,  False),
    ("nightly ETL with fixed steps",        False, True,  False),
    ("research an open question via tools",  False, False, True),
]
for label, single, det, flex in cases:
    print(f"  {label:38s} -> {decide(single, det, flex)}")

# Output:
#   translate one sentence                 -> just call the API once (no loop needed)
#   nightly ETL with fixed steps           -> use a WORKFLOW (you fix the steps)
#   research an open question via tools    -> use an AGENT (the LLM decides the path)

**Explanation**

A decision function that turns three facts about a task - single-shot, deterministic, needs runtime flexibility - into the right architecture. It operationalizes Anthropic's 'start simple, add agency only when needed' rule.

**How the code works - step by step**
- **`decide(single_shot, deterministic, needs_runtime_flexibility)`** - branches in priority order.
- **single-shot** -> just call the API once (no loop).
- **deterministic and not flexible** -> a workflow (you fix the steps).
- **needs runtime flexibility** -> an agent (the LLM decides the path).
- The `cases` list routes three tasks - one-sentence translation, nightly fixed-step ETL, open-ended tool research - through the function.

**In one line:** runtime flexibility is the trigger for an agent; most tasks are an API call or a workflow, and that is a feature.

**What you'll see:** Three lines: translate one sentence -> call the API once; nightly ETL with fixed steps -> use a workflow; research an open question via tools -> use an agent.

## 9 - One-flag agent: Google's SDK runs the loop for you

Every framework wraps the loop you just built by hand. The new `google-genai` SDK is the shortest example: pass your tools and automatic function calling (AFC) runs the whole think/act/observe loop internally - the model decides, calls, observes, and answers.

> **Needs a Gemini API key** (set `GEMINI_API_KEY`).

In [ ]:
# ONE-FLAG AGENT - the new google-genai SDK runs the loop for you (AFC on by default).
from google import genai
client = genai.Client()                              # reads GEMINI_API_KEY

def search_courses(topic: str) -> dict:
    "Search the Netsetos course catalog by topic."
    return {"genai": {"price": 14999}}.get(topic.lower(), {"error": "not found"})

resp = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="How much is the GenAI course?",
    config={"tools": [search_courses]},              # AFC runs the loop internally
)
print(resp.text)

# Output: (illustrative - needs `pip install google-genai` + a key) Gemini calls search_courses
#   for you and answers. Frameworks (OpenAI Agents SDK, LangGraph, CrewAI) wrap the same loop.
#   NOTE: the old `import google.generativeai` + enable_automatic_function_calling=True is deprecated.

**Explanation**

The framework payoff: a real Google SDK call where you hand over one Python function as a tool and the SDK's automatic function calling drives the loop for you. It closes the lesson by showing the same loop from cells 2 and 4, now fully abstracted away.

**How the code works - step by step**
- **`client = genai.Client()`** - the new `google-genai` client, reading `GEMINI_API_KEY`.
- **`search_courses(topic)`** - a plain typed Python function with a docstring; that is the entire tool definition.
- **`generate_content(..., config={"tools": [search_courses]})`** - passing the function turns on AFC, which runs the tool-calling loop internally.
- **`print(resp.text)`** - the model's final answer after it has called the tool and observed the result.
- The comment flags that the old `import google.generativeai` + `enable_automatic_function_calling=True` is deprecated.

**In one line:** pass a function as a tool and the SDK runs the whole agent loop for you.

**What you'll see:** Illustrative - with a valid key Gemini calls `search_courses` internally and prints an answer about the GenAI course price. The `# Output:` note stresses that OpenAI Agents SDK, LangGraph, and CrewAI all wrap this same loop, and that the old `google.generativeai` import is deprecated.

You have built the agent loop from scratch and seen the two things that matter most in practice: the loop is just think -> act -> observe repeated until a final answer, and every framework (Anthropic's `stop_reason` loop, Google's automatic function calling, OpenAI Agents SDK, LangGraph, CrewAI) wraps that same shape. The judgement pieces - ReAct reasoning before acting, `p^N` reliability that forces an iteration cap, history as short-term memory, and the rule to reach for an agent only on open-ended model-decided paths - are what separate a working agent from a fragile one. From here Module 11 goes deep: architectures (11.2), LangGraph (11.3), memory systems (11.4), multi-agent coordination (11.5), human-in-the-loop safety (11.10), and evaluation/security (11.11).